The Hall Effect was discovered by Edwin Hall by experimenting with Maxwell's equations of electricity and magnetism. He discovered that a magnetic field perpendicular to the direction of a current would result in small but measurable potential difference which was perpendicular again to both the direction of the current and the direction of the magnetic field. 

In the early 2000s, astrophysics scientists were finding that even weak magnetic fields present in collapsing stellar gas clouds could remove enough angular momentum from the system to totally prevent their collapse into stars. This became known as the magnetic braking catastrophe. In 2004, a paper from Mark Wardle ("Star Formation and the Hall Effect") showed that the Hall Effect should provide a mechanism for gas clouds to be able to collapse into stars despite the presence of stronger magnetic fields. 

In a spinning gas cloud, charged particles generate both an electric and magnetic field, and this can result in particles experiencing a force towards the center of the cloud.

To simulate this effect, I will use Python's pygame module to create a virtual space and populate it with a large number of particles. These particles will have masses and randomized directions, and some of them will be given charges. The particles will thus experience gravitational forces, causing the system to collapse, and hall effect forces based on the movements and locations of charges, which will generate both an electric and magnetic field. Instead of simulating an entire 3 dimensional gas cloud's evolution into a star, this will simulate a central x-y plane of the gas cloud. 

Each particle will have to calculate the forces it experiences from every other particle in the simulation, both magnetic and gravitational, for each time step of the simulation, so larger simulations will quickly become more computationally expensive.

# Relevant equations

## Electric field generated by a point particle:
$$ \overrightarrow{E} = \frac{1}{4 \pi \epsilon_0} \frac{q}{r^3} \overrightarrow{r} $$
Where:

$\epsilon_0 = 8.854$ $C^2$ $N$ $m^{-2}$ is the vacuum permittivity,

$q$ is the charge of the particle,

$r$ is the magnitude of the distance from the particle,

$\overrightarrow{r}$ is the position vector from the particle to the point of observation ($\overrightarrow{r}_{obs}-\overrightarrow{r}_{source}$)

### Electric force between two charged particles:
$$\overrightarrow{F_E} = q \overrightarrow{E}$$

## Magnetic field generated by moving charges:
$$\overrightarrow{B}=\frac{\mu_0}{4\pi}\frac{q \overrightarrow{v}\times\overrightarrow{r}}{\overrightarrow{r}^3}$$

Where:

$\mu_0 = 1.257 \times 10^{-7}$ $N s^2$ $C^{-2}$ is the vacuum permaebility,

$\overrightarrow{v}$ is the particle's velocity vector, and

$\overrightarrow{r}$ is the particle's position vector

### Magnetic Force
$$\overrightarrow{F_B} = q\overrightarrow{v} \times \overrightarrow{B}$$

## Lorentz Force
$$\overrightarrow{F_L} = q(\overrightarrow{E} + \overrightarrow{v} \times \overrightarrow{B}) $$

In order to simulate all of these effects then, every particle must know at each time step:

1. Its mass
2. Its position vector
3. Its velocity vector
4. Its charge

These four values can then be used to find the forces acting on each particle, allowing the position and velocity vectors to be updated the next time step. The first step will be to create functions for these equations which can be called while running pygame.

In [7]:
import numpy as np
import math
import pygame
import sys

# Define constants
G         = 6.674e-11 # N * m^2 * kg^-2
epsilon_0 = 8.85e-12  # C^2 N^-1 m^-2
mu_0      = 1.257e-7  # N s^2 C^-2

def calc_magnitude(vector):
    return np.linalg.norm(vector)

def calc_grav_force(m1, m2, r1_vec, r2_vec):
    """
    m1:     Mass of target object
    m2:     Mass of affecting object
    r1_vec: Position vector of target object
    r2_vec: Position vector of affecting object
    """
    r1_vec = np.asarray(r1_vec)
    r2_vec = np.asarray(r2_vec)
    d_vec  = r2_vec - r1_vec
    d_mag  = calc_magnitude(d_vec)
    return (G * m1 * m2) / d_mag**2

def calc_e_field(q, r_vec):
    r_vec = np.asarray(r_vec)
    r_mag = calc_magnitude(r_vec)
    return (1/ (4 * np.pi * epsilon_0)) * (q/r_mag**3)*r_vec

def calc_e_force(q, e_field):
    e_field = np.asarray(e_field)
    return q * e_field

def calc_b_field(q, v_vec, r_vec):
    v_vec = np.asarray(v_vec)
    r_vec = np.asarray(r_vec)
    r_mag = calc_magnitude(r_vec)
    return (mu_0 / (4 * np.pi)) * (q * (np.cross(v_vec, r_vec))/(r_mag **3))

def calc_b_force(q, v_vec, b_field):
    v_vec   = np.asarray(v_vec)
    b_field = np.asarray(b_field )
    return q * np.cross(v_vec, b_field)

# Running simulation

In [ ]:
pygame.init()

width, height = 1280, 720
win = pygame.display.set_mode((width, height))
pygame.display.set_caption("Star formation with Hall Effect")

# Color library
white        = (255,255,255)
black        = (0, 0, 0)
dark_gray    = (89, 89, 89)
yellow       = (255, 255, 0)
blue         = (0, 0, 255)
red          = (255, 0, 0)
cosmic_latte = (255, 248, 231)
gray         = (222, 222, 222)
light_blue   = (204,212,255)
brown        = (77,38,0)

# Font
font         = pygame.font.SysFont("ariel", 16)
pause_bttn   = pygame.Rect(20, 20, 100, 50)

class Particle:
    def __init__(self, x, y, radius, color, mass):
        self.x      = x
        self.y      = y
        self.radius = radius # in pixels
        self.color  = color
        self.mass   = mass

        self.x_vel           = 0 # m/s
        self.y_vel           = 0 # m/s
        self.z_vel           = 0 # m/s
        self.speed_magnitude = math.sqrt(self.x_vel**2 + self.y_vel**2 + self.z_vel**2)